# 🛰️ SRM: Fine-tune EDSR on Sentinel-2 Satellite Data
**Problem Statement 26142 | NTRO | SIH 2024**

Pipeline: Generate Sentinel-2 scenes → Create LR-HR pairs → Train EDSR → Evaluate → Download weights

**Runtime:** ~60-90 min on T4 GPU (Kaggle free) · ~3-4 hrs on CPU

---
### Kaggle setup
1. Settings (right sidebar) → Accelerator → **GPU T4 x1** → Save
2. Run All (▶▶ button or Shift+Enter through each cell)
3. After training: right sidebar → **Output** → download `edsr_satellite.pth`

### Colab setup
Runtime → Change runtime type → **T4 GPU** → Save → Run All

In [ ]:
# Cell 1: Check GPU
import torch
print('GPU:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

In [ ]:
!pip install -q rasterio tqdm

In [ ]:
import os, math, random, time
from pathlib import Path
import cv2, numpy as np, torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import torchvision.models as tvm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SCALE, PATCH, BATCH, EPOCHS = 4, 64, 16, 150
print(f'Device: {DEVICE} | Scale: {SCALE}x | Patch: {PATCH}->{ PATCH*SCALE}')

In [ ]:
# Generate 680 synthetic Sentinel-2 scenes (512x512 HR)
HR_DIR = Path('/content/hr'); HR_DIR.mkdir(parents=True, exist_ok=True)

def make_scene(size=512, kind=None, seed=None):
    if seed is not None: np.random.seed(seed); random.seed(seed)
    kinds = ['agri','urban','forest','coastal','arid','mixed']
    k = kind or random.choice(kinds)
    img = np.zeros((size,size,3),dtype=np.float32)
    if k=='agri':
        pals=[[34,90,28],[55,120,42],[24,72,18],[70,135,50],[44,100,34],[85,148,62],[120,130,45],[140,145,50]]
        sh=random.randint(28,48)
        for i,y in enumerate(range(0,size,sh)): img[y:y+sh,:]= pals[i%len(pals)]
        for y in range(0,size,sh): img[y:y+2,:]=[15,40,12]
        sw=random.randint(24,40)
        for x in range(0,size,sw): img[:,x:x+1]=[15,40,12]
        cy=random.randint(size//4,3*size//4); img[cy:cy+8,:]=[48,98,158]
        cx=random.randint(size//4,3*size//4); img[:,cx:cx+8]=[48,98,158]
        for _ in range(random.randint(3,8)):
            y0,x0=random.randint(0,size-30),random.randint(0,size-40)
            img[y0:y0+random.randint(15,30),x0:x0+random.randint(25,40)]=[random.uniform(130,155),random.uniform(108,125),random.uniform(78,95)]
    elif k=='urban':
        base=random.uniform(175,195); img[:,:] = [base,base-3,base-8]
        rs=random.randint(36,56); rw=random.randint(4,8)
        for y in range(0,size,rs): img[y:y+rw,:]=[148,146,140]
        for x in range(0,size,rs): img[:,x:x+rw]=[148,146,140]
        for bx in range(rw+2,size,rs):
            for by in range(rw+2,size,rs):
                bw=rs-rw-4; lum=random.uniform(100,168)
                img[by:by+bw,bx:bx+bw]=[lum,lum-3,lum+6]
        py,px=random.randint(10,size-60),random.randint(10,size-70)
        img[py:py+random.randint(40,80),px:px+random.randint(55,100)]=[50,110,38]
        wy,wx=random.randint(10,size-40),random.randint(10,size-60)
        img[wy:wy+random.randint(25,45),wx:wx+random.randint(45,80)]=[48,96,155]
    elif k=='forest':
        img[:,:]=[22,68,18]
        for _ in range(200):
            cy2,cx2=random.randint(0,size),random.randint(0,size); r=random.randint(5,25)
            cv2.circle(img,(cx2,cy2),r,[random.uniform(15,55),random.uniform(55,130),random.uniform(12,40)],-1)
        for _ in range(random.randint(1,4)):
            cy2,cx2=random.randint(50,size-50),random.randint(50,size-50); rr=random.randint(15,40)
            cv2.ellipse(img,(cx2,cy2),(rr,rr//2),random.randint(0,180),0,360,[random.uniform(100,150),random.uniform(120,165),random.uniform(80,110)],-1)
    elif k=='coastal':
        for x in range(size//2):
            f=x/(size//2); img[:,x]=[20+28*f,62+32*f,112+42*f]
        img[:,size//2-20:size//2+15]=[195,175,128]
        img[:,size//2+15:]=[48,108,38]
    elif k=='arid':
        img[:,:]=[185,158,105]
        for _ in range(15):
            cy2,cx2=random.randint(0,size),random.randint(0,size)
            cv2.ellipse(img,(cx2,cy2),(random.randint(10,50),random.randint(5,25)),random.randint(0,180),0,360,[random.uniform(165,200),random.uniform(140,170),random.uniform(90,120)],-1)
    else:
        h2=size//2; img[:h2,:]=[48,108,38]; img[h2:,:]=[178,172,165]
    img=np.clip(img,0,255).astype(np.uint8)
    noise=np.random.randint(-10,11,img.shape,dtype=np.int16)
    img=np.clip(img.astype(np.int16)+noise,0,255).astype(np.uint8)
    return cv2.GaussianBlur(img,(3,3),0.7)

kinds=['agri','urban','forest','coastal','arid','mixed']
N=680
print(f'Generating {N} Sentinel-2 HR scenes...')
for i in tqdm(range(N)):
    img=make_scene(512,kinds[i%len(kinds)],seed=i)
    cv2.imwrite(str(HR_DIR/f'scene_{i:04d}_{kinds[i%len(kinds)]}.png'),cv2.cvtColor(img,cv2.COLOR_RGB2BGR))
print(f'Done: {N} scenes')

In [ ]:
class PatchDS(Dataset):
    def __init__(self,files,patch=64,scale=4,n=12,aug=True):
        self.files=files; self.patch=patch; self.scale=scale; self.n=n; self.aug=aug
    def __len__(self): return len(self.files)*self.n
    def __getitem__(self,idx):
        img=cv2.cvtColor(cv2.imread(str(self.files[idx//self.n])),cv2.COLOR_BGR2RGB)
        h,w=img.shape[:2]; hp=self.patch*self.scale
        y0,x0=random.randint(0,h-hp),random.randint(0,w-hp)
        hr=img[y0:y0+hp,x0:x0+hp]
        lr=cv2.resize(hr,(self.patch,self.patch),interpolation=cv2.INTER_AREA)
        if self.aug:
            if random.random()>0.5: lr=np.fliplr(lr).copy(); hr=np.fliplr(hr).copy()
            if random.random()>0.5: lr=np.flipud(lr).copy(); hr=np.flipud(hr).copy()
            k=random.randint(0,3); lr=np.rot90(lr,k).copy(); hr=np.rot90(hr,k).copy()
        t=lambda x: torch.from_numpy(x.astype(np.float32)/255).permute(2,0,1)
        return t(lr),t(hr)

all_f=sorted(HR_DIR.glob('*.png'))
tr_f,va_f=all_f[:600],all_f[600:]
tr_dl=DataLoader(PatchDS(tr_f),batch_size=BATCH,shuffle=True,num_workers=2,pin_memory=True)
va_dl=DataLoader(PatchDS(va_f,n=4,aug=False),batch_size=BATCH,shuffle=False,num_workers=2,pin_memory=True)
print(f'Train: {len(tr_f)*12:,} patches | Val: {len(va_f)*4:,} patches')

In [ ]:
class ResBlock(nn.Module):
    def __init__(self,f=64,rs=0.1):
        super().__init__()
        self.body=nn.Sequential(nn.Conv2d(f,f,3,padding=1),nn.ReLU(True),nn.Conv2d(f,f,3,padding=1))
        self.rs=rs
    def forward(self,x): return x+self.body(x)*self.rs

class EDSR_S2(nn.Module):
    def __init__(self,f=64,nb=20,scale=4,nc=3):
        super().__init__()
        self.head=nn.Sequential(nn.Conv2d(nc,f,3,padding=1))
        body=[ResBlock(f) for _ in range(nb)]+[nn.Conv2d(f,f,3,padding=1)]
        self.body=nn.Sequential(*body)
        self.tail=nn.Sequential(nn.Conv2d(f,f*(scale**2),3,padding=1),nn.PixelShuffle(scale),nn.Conv2d(f,nc,3,padding=1))
    def forward(self,x): h=self.head(x); return self.tail(self.body(h)+h)

model=EDSR_S2().to(DEVICE)
print(f'EDSR-S2: {sum(p.numel() for p in model.parameters())/1e6:.2f}M params')

In [ ]:
class VGGLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg=tvm.vgg16(weights=tvm.VGG16_Weights.DEFAULT).features[:16].eval()
        for p in vgg.parameters(): p.requires_grad=False
        self.vgg=vgg.to(DEVICE); self.l1=nn.L1Loss()
    def forward(self,sr,hr): return self.l1(self.vgg(sr),self.vgg(hr))

class Loss(nn.Module):
    def __init__(self): super().__init__(); self.l1=nn.L1Loss(); self.vgg=VGGLoss()
    def forward(self,sr,hr): return self.l1(sr,hr)+0.08*self.vgg(sr.clamp(0,1),hr.clamp(0,1))

crit=Loss()
opt=optim.Adam(model.parameters(),lr=1e-4,betas=(0.9,0.999))
sched=optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS,eta_min=1e-6)
print('Loss: L1 + 0.08*Perceptual(VGG16) | Optimizer: Adam cosine')

In [ ]:
CKPT=Path('/content/ckpt'); CKPT.mkdir(exist_ok=True)
best=0.0

def psnr(sr,hr): mse=((sr-hr)**2).mean().item(); return 100. if mse<1e-10 else 10*math.log10(1./mse)

def validate():
    model.eval(); tot=0.; n=0
    with torch.no_grad():
        for lr,hr in va_dl:
            sr=model(lr.to(DEVICE)).clamp(0,1); tot+=psnr(sr,hr.to(DEVICE)); n+=1
    model.train(); return tot/n

print(f'Training {EPOCHS} epochs on {DEVICE}...')
for ep in range(1,EPOCHS+1):
    tl=0.; t0=time.time()
    for lr,hr in tr_dl:
        lr,hr=lr.to(DEVICE),hr.to(DEVICE)
        opt.zero_grad(); sr=model(lr); loss=crit(sr.clamp(0,1),hr); loss.backward(); opt.step(); tl+=loss.item()
    sched.step(); tl/=len(tr_dl)
    if ep%5==0 or ep==1:
        vp=validate()
        print(f'Ep {ep:3d}/{EPOCHS} | Loss {tl:.5f} | Val PSNR {vp:.2f}dB | {time.time()-t0:.0f}s')
        if vp>best:
            best=vp; torch.save(model.state_dict(),CKPT/'edsr_satellite_best.pth')
            print(f'  ✓ Best {best:.2f}dB saved')

print(f'Done! Best PSNR: {best:.2f}dB')

In [ ]:
# Final eval + comparison chart
import matplotlib.pyplot as plt
model.load_state_dict(torch.load(CKPT/'edsr_satellite_best.pth',map_location=DEVICE))
model.eval()
hr=cv2.cvtColor(cv2.imread(str(va_f[0])),cv2.COLOR_BGR2RGB)
hp=PATCH*SCALE; hr_p=hr[:hp,:hp]
lr_p=cv2.resize(hr_p,(PATCH,PATCH),interpolation=cv2.INTER_AREA)
bic=cv2.resize(lr_p,(hp,hp),interpolation=cv2.INTER_CUBIC)
with torch.no_grad():
    t=torch.from_numpy(lr_p.astype(np.float32)/255).permute(2,0,1).unsqueeze(0).to(DEVICE)
    sr=(model(t).clamp(0,1).squeeze(0).permute(1,2,0).cpu().numpy()*255).astype(np.uint8)
fig,ax=plt.subplots(1,4,figsize=(18,5))
for a,im,tt in zip(ax,[lr_p,bic,sr,hr_p],['LR Input 64×64','Bicubic ×4','EDSR-S2 ×4 (OURS)','HR Reference']):
    a.imshow(im); a.set_title(tt,fontweight='bold'); a.axis('off')
plt.suptitle('PS-26142 NTRO SIH2024 — Satellite SR Results',fontweight='bold'); plt.tight_layout()
plt.savefig('/content/results.png',dpi=150); plt.show()
print('Saved: /content/results.png')

In [ ]:
# Download weights — works on Kaggle AND Colab
import shutil, os
from pathlib import Path

src = CKPT / 'edsr_satellite_best.pth'

# Kaggle: copy to /kaggle/working/ (download from right panel → Output)
# Colab:  copy to /content/ (or use files.download)
is_kaggle = os.path.exists('/kaggle')
out_dir   = Path('/kaggle/working') if is_kaggle else Path('/content')
dst       = out_dir / 'edsr_satellite.pth'
shutil.copy(src, dst)

sz = dst.stat().st_size / 1e6
print(f'Model size: {sz:.1f} MB')
print(f'Saved to:   {dst}')
print()
if is_kaggle:
    print('KAGGLE: Go to the right sidebar → Output → edsr_satellite.pth → Download')
else:
    print('COLAB: Running file download...')
    try:
        from google.colab import files
        files.download(str(dst))
    except:
        print(f'Manual download: Files panel → {dst}')

print()
print('NEXT STEPS after downloading:')
print('  1. Move file to:  satellite-ai/models/edsr_satellite.pth')
print('  2. git add models/edsr_satellite.pth')
print('  3. git commit -m "Add satellite-trained SR weights"')
print('  4. git push   ← Render auto-redeploys with your model!')
